In [0]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import DateType, DoubleType

In [0]:
# Cluster Spark
spark = SparkSession.builder.getOrCreate()

In [0]:
# lê da bronze e converte para pandas
df_bronze = spark.table('db_analytics.precos_brutos_bitcoin').toPandas()
type(df_bronze)

In [0]:
# Verificar
print(df_bronze.shape)
print(df_bronze.dtypes)
df_bronze.sample(5)

In [0]:
# Converter data
df_bronze['data'] = pd.to_datetime(df_bronze['data'], errors='coerce')

In [0]:
# define início da semana
df_bronze['semana_inicio'] = (
    df_bronze['data']
    .dt.to_period('W')
    .apply(lambda loop: loop.start_time)
)

df_bronze.head()


In [0]:
# Analise da quantidade de registros na semana
(
    df_bronze
    .semana_inicio
    .value_counts()
    .sort_index()
)

In [0]:
# agrega por semana
df_gold = (
    df_bronze
    .groupby('semana_inicio')
    .agg(
        semana_fim   = ('data',       'max'),
        abertura     = ('abertura',   'first'),
        fechamento   = ('fechamento', 'last'),
        maxima       = ('maxima',     'max'),
        minima       = ('minima',     'min'),
        volume_medio = ('volume',     'mean')
    )
    .reset_index()
)

df_gold.head()

In [0]:
# calcula variação percentual
df_gold['variacao_pct'] = ((df_gold['fechamento'] - df_gold['abertura']) / df_gold['abertura'] * 100).round(2)

df_gold.head()


In [0]:
# corrige tipos
df_gold['semana_inicio'] = pd.to_datetime(df_gold['semana_inicio']).dt.date
df_gold['semana_fim']    = pd.to_datetime(df_gold['semana_fim']).dt.date

In [0]:
# converte DF para Spark e salva
df_spark = spark.createDataFrame(df_gold)

(
    df_spark
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable('db_analytics.precos_semanal_bitcoin')
)

print(f'Total de semanas processadas: {df_spark.count()}')